# Viewer

Generation + attribution now run via the standalone scripts
`experiment/generation-gemma-3-{270m,1b,4b}.py` (looping over the
5-prompt set in `tools/prompt_set.json`), not in this notebook.
Run the appropriate script first to populate `./graphs/gemma-3-<size>/<slug>/`,
then use the cells below to browse a given prompt's graphs.

In [ ]:
from pathlib import Path

# One of: "original", "hard", "medium", "easy", "repetition_control"
slug = "original"
graph_dir = Path(f"./graphs/gemma-3-270m/{slug}")

In [ ]:
import json
from pathlib import Path

for json_path in sorted(graph_dir.glob("step-*.json")):
    data = json.loads(json_path.read_text())
    
    # collect all unique layers used by transcoder nodes
    layers = sorted(set(
        int(node["layer"])
        for node in data["nodes"]
        if not node["is_target_logit"] and node.get("feature_type") == "cross layer transcoder"
    ))
    
    transcoder_list = [f"gemma-2-2b/{l}-gemmascope-transcoder-16k" for l in layers]
    data["metadata"]["transcoder_list"] = transcoder_list
    
    json_path.write_text(json.dumps(data))
    print(f"{json_path.name}: {len(layers)} layers → {transcoder_list[:3]}...")

# also patch the manifest
manifest_path = graph_dir / "graph-metadata.json"
manifest = json.loads(manifest_path.read_text())
for entry in manifest["graphs"]:
    slug = entry["slug"]
    step_path = graph_dir / f"{slug}.json"
    if step_path.exists():
        step_data = json.loads(step_path.read_text())
        entry["transcoder_list"] = step_data["metadata"]["transcoder_list"]

manifest_path.write_text(json.dumps(manifest, indent=2))
print("manifest patched")

In [ ]:
for json_path in sorted(graph_dir.glob("step-*.json")):
    data = json.loads(json_path.read_text())
    layers = sorted(set(
        int(node["layer"])
        for node in data["nodes"]
        if not node["is_target_logit"] and node.get("feature_type") == "cross layer transcoder"
    ))
    # correct 1B slug
    data["metadata"]["transcoder_list"] = [
        f"gemma-3-270m/{l}-gemmascope-2-transcoder-16k" for l in layers
    ]
    json_path.write_text(json.dumps(data))
    print(f"{json_path.name}: patched with 1B slugs")

# patch manifest too
manifest_path = graph_dir / "graph-metadata.json"
manifest = json.loads(manifest_path.read_text())
for entry in manifest["graphs"]:
    slug = entry["slug"]
    step_path = graph_dir / f"{slug}.json"
    if step_path.exists():
        step_data = json.loads(step_path.read_text())
        entry["transcoder_list"] = step_data["metadata"]["transcoder_list"]

manifest_path.write_text(json.dumps(manifest, indent=2))
print("manifest repaired")

In [ ]:
import json
from pathlib import Path

data = {"graphs": []}  # ← Add this line to initialize

for json_path in sorted(graph_dir.glob("step-*.json")):
    graph_data = json.loads(json_path.read_text())
    data["graphs"].append(graph_data["metadata"])

(graph_dir / "graph-metadata.json").write_text(json.dumps(data, indent=2))
print("done")

In [ ]:
from circuit_tracer.frontend.local_server import serve
from IPython.display import IFrame

port = 8046
server = serve(data_dir=str(graph_dir), port=port)

print(f"http://localhost:{port}/index.html")
display(IFrame(src=f"http://localhost:{port}/index.html", width="100%", height="800px"))